# R-Sparse Standalone Pipeline (Colab T4, Drive-persistent)

Self-contained replacement for the `project/` scripts. Every cell inlines the logic of one script — there's nothing extra to upload, and **everything that takes time persists on Google Drive**, so a Colab session reset only costs you the pip-install (~3-5 min) and not the SVD precompute or model download.

**Drive layout used by this notebook:**
```
/content/drive/MyDrive/r_sparse/
├── repo/                    # cloned R-Sparse (no re-clone on reconnect)
├── hf_cache/                # HuggingFace downloads (6 GB Llama-3.2 weights, etc.)
├── low_rank_models/<alias>/ # SVD .pt files
├── configs/                 # JSON configs
└── results/                 # baseline + r-sparse + summary JSONs
```

**Pipeline:** SVD precompute → R-Sparse JSON config → HF baseline (accuracy + latency) → R-Sparse eval (accuracy + latency) → comparison table.

**Default model:** `meta-llama/Llama-3.2-3B` (best fit for a Colab T4 — ~6 GB FP16, 28 layers, pure Llama architecture). Switch in §8.

**Why HF instead of vLLM for the baseline:** baseline and R-Sparse both go through `lm-eval` with HF generate, so the comparison is apples-to-apples (same engine, same evaluator). vLLM has notebook-cleanup pitfalls (it holds GPU memory until process exit). An optional vLLM-latency cell sits at the end of the notebook if you want a separate vLLM speed reference.

---

## §1 Mount Google Drive

Required for persistence. You'll get a popup to authorize Drive access — accept it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## §2 Sanity-check the runtime

Make sure you're on a GPU runtime: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU'
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
!free -h | head -2

## §3 Configure Drive-persistent paths

Every artifact that costs >1 minute to recreate goes under `MyDrive/r_sparse/`. The `os.environ['HF_HOME']` line is what makes HuggingFace cache model weights on Drive — without it Llama-3.2-3B (6 GB) would re-download every session.

In [ ]:
import os
from pathlib import Path

PERSIST_ROOT = Path('/content/drive/MyDrive/r_sparse')
REPO_DIR     = PERSIST_ROOT / 'repo'
HF_CACHE     = PERSIST_ROOT / 'hf_cache'
SVD_ROOT     = PERSIST_ROOT / 'low_rank_models'
CONFIG_ROOT  = PERSIST_ROOT / 'configs'
RESULTS_DIR  = PERSIST_ROOT / 'results'

for d in (PERSIST_ROOT, HF_CACHE, SVD_ROOT, CONFIG_ROOT, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Route HuggingFace caches to Drive (model weights, datasets, tokenizers).
os.environ['HF_HOME']             = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']   = str(HF_CACHE / 'datasets')
os.environ['TRANSFORMERS_CACHE']  = str(HF_CACHE / 'transformers')

for k in ('HF_HOME', 'HF_DATASETS_CACHE', 'TRANSFORMERS_CACHE'):
    print(f'{k:22s} = {os.environ[k]}')
print(f'\nPERSIST_ROOT = {PERSIST_ROOT}')
print(f'REPO_DIR     = {REPO_DIR}')
print(f'SVD_ROOT     = {SVD_ROOT}')
print(f'RESULTS_DIR  = {RESULTS_DIR}')

## §4 Clone R-Sparse to Drive (or reuse existing clone)

We need the upstream repo for two imports — `utils.setup.setup_model` and `models.modeling_llama.LlamaForCausalLM_R_Sparse`. Cloning to Drive means it survives session resets. **Nothing in R-Sparse is modified.**

In [ ]:
# If you've pushed a fork that contains the project/ scripts, switch the URL.
REPO_URL = 'https://github.com/Zhang-AIs-AI/R-Sparse.git'

if not (REPO_DIR / '.git').exists():
    print(f'Cloning {REPO_URL} → {REPO_DIR}')
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    print(f'Reusing existing clone at {REPO_DIR}')

%cd $REPO_DIR
!ls

## §5 Install dependencies (per-session — pip caches don't persist on Drive)

We bump `transformers` past R-Sparse's pin (4.39.2 → 4.45) so Llama-3.2's `rope_scaling: {rope_type: 'llama3'}` is recognized. R-Sparse's `modeling_llama.py` only uses stable APIs (`LlamaForCausalLM`, `apply_rotary_pos_emb`, `repeat_kv`), so the bump is safe.

Persisting the pip site-packages on Drive is fragile (Python version drift, native binary mismatches), so we re-install each session. Takes ~3-5 minutes.

In [ ]:
%pip install -q --upgrade pip
%pip install -q \
    'transformers>=4.45,<4.50' \
    'accelerate>=0.34' \
    'datasets>=2.14' \
    'sentencepiece' 'scipy' 'scikit-learn' 'tqdm' 'pandas' 'numpy<2.0' \
    'huggingface_hub'
# vllm + lm-eval are optional — only needed for the optional vLLM-latency cell at §17.
%pip install -q 'vllm>=0.6.0' 'lm-eval[vllm]>=0.4.4'

## §6 HuggingFace login

Llama-3.2-3B, Llama-3-8B-Instruct, and Mistral-7B-v0.3 are gated. Get a Read token at https://huggingface.co/settings/tokens, accept the model license at e.g. https://huggingface.co/meta-llama/Llama-3.2-3B.

In [ ]:
from huggingface_hub import login
login()

## §7 Model registry (inline replacement for `registry.py`)

Three target models. R-Sparse-compatibility flag is currently `True` for all three; the field is kept so a fused-projection model (e.g. Phi-3) could be re-added with one line.

In [ ]:
MODEL_REGISTRY = {
    'llama32_3b': {
        'hf_id': 'meta-llama/Llama-3.2-3B',
        'num_layers': 28,
        'supports_rsparse': True,
        'note': 'Best fit for T4. Pure Llama arch; needs transformers>=4.43 for new rope_scaling.',
        'vram_fp16_gb': 6.0,
    },
    'llama3': {
        'hf_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
        'num_layers': 32,
        'supports_rsparse': True,
        'note': 'Canonical R-Sparse target — what scripts/example.sh exercises. Tight on T4 VRAM.',
        'vram_fp16_gb': 16.0,
    },
    'mistral': {
        'hf_id': 'mistralai/Mistral-7B-v0.3',
        'num_layers': 32,
        'supports_rsparse': True,
        'note': 'Loaded through LlamaForCausalLM_R_Sparse; sliding-window dropped (v0.3 disables it by default).',
        'vram_fp16_gb': 14.0,
    },
}

## §8 Choose the model + task

Edit this cell to switch models. Default is Llama-3.2-3B with a 64-example PIQA smoke test. Set `LIMIT = None` for the full eval (~1.8k examples — adds maybe 30 min on T4).

In [ ]:
MODEL_ALIAS = 'llama32_3b'   # llama32_3b | llama3 | mistral
TASK        = 'piqa'
LIMIT       = 64             # int (smoke test) or None (full eval)

# R-Sparse hyperparameters (paper-recommended starting point)
TARGET_SPARSITY = 0.5
SPARSE_RATIO    = 0.5     # 0=pure low-rank, 1=pure sparse, 0.5=hybrid
PREFILL_RATIO   = 0.1

entry = MODEL_REGISTRY[MODEL_ALIAS]
HF_ID = entry['hf_id']
NUM_LAYERS = entry['num_layers']
print(f'Model:        {HF_ID}')
print(f'Layers:       {NUM_LAYERS}')
print(f'R-Sparse OK:  {entry["supports_rsparse"]}')
print(f'Note:         {entry["note"]}')
print(f'\nTask: {TASK}, limit: {LIMIT}')
print(f'R-Sparse: target_sparsity={TARGET_SPARSITY}, sparse_ratio={SPARSE_RATIO}, prefill_ratio={PREFILL_RATIO}')

## §9 Tokenizer monkey-patch (Llama-3.x compatibility)

R-Sparse's `utils/setup.py` calls `AutoTokenizer.from_pretrained(..., use_fast=False)`. Llama-3.x has no SentencePiece tokenizer (only `tokenizer.json`), so `use_fast=False` would error. We wrap `AutoTokenizer.from_pretrained` to ignore that flag for this session — R-Sparse internals stay unmodified.

In [ ]:
import transformers
import functools

_orig_tok_fp = transformers.AutoTokenizer.from_pretrained

@functools.wraps(_orig_tok_fp)
def _tok_fp_wrapper(*args, **kwargs):
    kwargs['use_fast'] = True
    return _orig_tok_fp(*args, **kwargs)

transformers.AutoTokenizer.from_pretrained = _tok_fp_wrapper
print('AutoTokenizer.from_pretrained is now forced to use_fast=True')

## §10 SVD precompute (inline replacement for `prepare_svd.py` + `utils/prepare_low_rank_weight.py`)

For every `nn.Linear` in the model, decompose `W = U·diag(S)·Vᵀ` and save `(U, S, V, scale)` to `<SVD_DIR>/<module_name>.pt`. R-Sparse uses these at inference to combine sparse + low-rank paths.

**Why we load with `low_cpu_mem_usage=False, device_map=None`:** if `accelerate` puts any weight on a *meta* tensor (placeholder without data), the line `weight.data.to(torch.float64).cuda()` raises `NotImplementedError: Cannot copy out of meta tensor` partway through. Forcing full materialization to CPU avoids this.

**Resume support:** files already in `SVD_DIR` are skipped. Safe to re-run after a disconnect.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm
import gc

SVD_DIR = SVD_ROOT / MODEL_ALIAS
SVD_DIR.mkdir(parents=True, exist_ok=True)
print(f'SVD output: {SVD_DIR}')

existing = {p.stem for p in SVD_DIR.glob('*.pt')}
print(f'Already-computed layers: {len(existing)} (will be skipped)')

print(f'\nLoading {HF_ID} (fp16, full CPU materialisation)...')
svd_model = AutoModelForCausalLM.from_pretrained(
    HF_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=False,   # avoid meta-tensor offload
    device_map=None,
    cache_dir=str(HF_CACHE),
)

linears = [(name, m) for name, m in svd_model.named_modules() if isinstance(m, nn.Linear)]
print(f'Total linear layers: {len(linears)}')

for name, m in tqdm(linears, desc='SVD'):
    out_path = SVD_DIR / f'{name}.pt'
    if name in existing:
        continue

    weight_gpu = m.weight.data.to(torch.float64).cuda()
    u, s, v = torch.svd(weight_gpu)
    error = torch.norm(weight_gpu - u @ torch.diag(s) @ v.T).item()

    u_h = u.to(torch.float16).cpu()
    s_h = s.to(torch.float16).cpu()
    v_h = v.to(torch.float16).cpu()
    scale = (v_h.to(torch.float64) @ torch.diag(s_h.to(torch.float64))).norm(dim=1).to(torch.float16)
    torch.save((u_h, s_h, v_h, scale), out_path)

    del weight_gpu, u, s, v, u_h, s_h, v_h, scale
    torch.cuda.empty_cache()

del svd_model
gc.collect(); torch.cuda.empty_cache()
print(f'\nDone. {len(list(SVD_DIR.glob("*.pt")))} .pt files at {SVD_DIR}')

## §11 Build the R-Sparse JSON config (inline replacement for `prepare_svd.py:build_config`)

R-Sparse's `setup_config` reads per-projection paths from this JSON. Per-layer `threshold` / `low_rank` values here are placeholders — the real numbers are recomputed by `set_threshold_r_sparse` from the budget formula at calibration time.

In [ ]:
import json

PROJ_TO_SUB = {
    'q':    'self_attn',
    'k':    'self_attn',
    'v':    'self_attn',
    'o':    'self_attn',
    'gate': 'mlp',
    'up':   'mlp',
    'down': 'mlp',
}

config = {}
for proj, sub in PROJ_TO_SUB.items():
    paths = [str(SVD_DIR / f'model.layers.{i}.{sub}.{proj}_proj.pt') for i in range(NUM_LAYERS)]
    config[f'{proj}_svd_path']   = paths
    config[f'{proj}_threshold']  = [0.0] * NUM_LAYERS
    config[f'{proj}_low_rank']   = [128] * NUM_LAYERS

CONFIG_PATH = CONFIG_ROOT / f'{MODEL_ALIAS}_default.json'
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f)
print(f'Wrote {CONFIG_PATH}')
print(f'  {len(config)} top-level keys, {NUM_LAYERS} layers each')

## §12 Shared latency prompts

Same 8 prompts used for the baseline and R-Sparse runs, so the speedup comparison is apples-to-apples (same workload, same engine, only the model is different).

In [ ]:
LATENCY_PROMPTS = [
    'Explain quantum entanglement to a high-school student in two sentences.',
    'Write a short Python function that returns the n-th Fibonacci number.',
    'Summarize the plot of Hamlet in three sentences.',
    'List five risks of deploying an LLM as a customer-support agent.',
    "Translate to French: 'The quick brown fox jumps over the lazy dog.'",
    "What is the capital of Australia, and why isn't it Sydney?",
    'Give one real-world example of a Markov chain.',
    'Write a haiku about debugging.',
]
LATENCY_MAX_TOKENS = 64

## §13 Helper: load + evaluate + time a model (used by §14 baseline and §15 R-Sparse)

One function so baseline and R-Sparse run identical code paths — same evaluator (`lm_eval.evaluator.simple_evaluate` from R-Sparse's bundled fork), same latency loop. The only difference is `args.method` (`'full'` for baseline, `'r_sparse'` for R-Sparse).

In [ ]:
import sys, time, json, gc
from types import SimpleNamespace
import torch

# Make R-Sparse importable. Doing this once for the rest of the notebook.
sys.path.insert(0, str(REPO_DIR))
from utils.setup import setup_model
from lm_eval import tasks as lm_tasks, evaluator as lm_evaluator, utils as lm_utils


def run_method(method, *, target_sparsity=0.5, sparse_ratio=1.0, prefill_ratio=1.0):
    """Load the model under the given method, run lm-eval accuracy on TASK,
    measure latency on LATENCY_PROMPTS, return a result dict (also written
    to RESULTS_DIR/{MODEL_ALIAS}_{tag}.json). Frees GPU after.
    """
    tag = 'baseline' if method == 'full' else 'rsparse' if method == 'r_sparse' else method
    args = SimpleNamespace(
        model_name        = HF_ID,
        cache_dir         = str(HF_CACHE),
        device            = 'cuda:0',
        method            = method,
        target_sparsity   = target_sparsity,
        sparse_ratio      = sparse_ratio,
        prefill_ratio     = prefill_ratio,
        config_file       = str(CONFIG_PATH),
        sparse_config_file = None,
    )

    print(f'\n=== {tag} ({method}) ===')
    print(f'Loading {HF_ID}...')
    _, tokenizer, model = setup_model(args)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = model.eval().to('cuda:0')

    # ---- Accuracy via R-Sparse's bundled lm_eval ----
    task_names = lm_utils.pattern_match([TASK], lm_tasks.ALL_TASKS)
    print(f'Evaluating on tasks: {task_names}')
    eval_results = lm_evaluator.simple_evaluate(
        model        = model,
        tasks        = task_names,
        num_fewshot  = 0,
        batch_size   = 1,
        device       = 'cuda:0',
        no_cache     = True,
        limit        = LIMIT,
        tokenizer    = tokenizer,
    )
    print(lm_evaluator.make_table(eval_results))

    # ---- Latency on the shared prompt set ----
    enc = tokenizer(LATENCY_PROMPTS, return_tensors='pt', padding=True).to('cuda:0')
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=4, do_sample=False)   # warm-up
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=LATENCY_MAX_TOKENS, do_sample=False)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    new_tokens = (out.shape[1] - enc['input_ids'].shape[1]) * out.shape[0]
    tps = new_tokens / elapsed if elapsed > 0 else 0.0
    print(f'Latency: {tps:.2f} tok/s  ({new_tokens} tokens in {elapsed:.2f} s)')

    # ---- Persist ----
    result = {
        'alias'           : MODEL_ALIAS,
        'hf_id'           : HF_ID,
        'task'            : TASK,
        'limit'           : LIMIT,
        'method'          : method,
        'target_sparsity' : target_sparsity if method == 'r_sparse' else None,
        'sparse_ratio'    : sparse_ratio    if method == 'r_sparse' else None,
        'prefill_ratio'   : prefill_ratio   if method == 'r_sparse' else None,
        'accuracy'        : {
            'results'  : eval_results.get('results'),
            'versions' : eval_results.get('versions'),
        },
        'latency'         : {
            'wall_clock_s'        : elapsed,
            'total_output_tokens' : int(new_tokens),
            'tokens_per_sec'      : tps,
            'num_prompts'         : len(LATENCY_PROMPTS),
            'max_new_tokens'      : LATENCY_MAX_TOKENS,
        },
    }
    out_path = RESULTS_DIR / f'{MODEL_ALIAS}_{tag}.json'
    out_path.write_text(json.dumps(result, indent=2, default=str))
    print(f'Wrote {out_path}')

    # ---- Free GPU before the next stage ----
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result

## §14 Baseline run (HF, full / unmodified model)

Replaces `run_baseline.py`. Loads the model with `setup_model(method='full')` — that's R-Sparse's own way of returning the vanilla `AutoModelForCausalLM` (no sparsity wrapping). Then accuracy + latency through the same code path R-Sparse will use, so the comparison is fair.

In [ ]:
baseline = run_method('full')

## §15 R-Sparse run

Replaces `run_rsparse.py`. `setup_model(method='r_sparse')` swaps every linear in `self_attn` / `mlp` for `R_Sparse_Linear`, loads the SVD `.pt` files referenced by the JSON config, and runs the wikitext-2 calibration pass to fill in per-layer thresholds (you'll see one `Setting threshold: ... Estimated sparsity: ...` line per layer × projection).

In [ ]:
rsparse = run_method(
    'r_sparse',
    target_sparsity = TARGET_SPARSITY,
    sparse_ratio    = SPARSE_RATIO,
    prefill_ratio   = PREFILL_RATIO,
)

## §16 Comparison table (inline replacement for `benchmark.py`)

Reads the two JSONs you just wrote. The table shows: baseline vs R-Sparse accuracy on TASK, the absolute Δ, baseline vs R-Sparse tok/s on the shared prompt set, and the R-Sparse / baseline speed ratio. A machine-readable `summary.json` is also written.

In [ ]:
import pandas as pd

def extract_acc(report, task):
    raw = report.get('accuracy') if report else None
    results = raw.get('results') if isinstance(raw, dict) else None
    if not results:
        return None
    metrics = results.get(task) or next(iter(results.values()), {})
    for key in ('acc,none', 'acc', 'exact_match', 'accuracy'):
        if key in metrics:
            return metrics[key]
    for v in metrics.values():
        if isinstance(v, (int, float)):
            return v
    return None

baseline_acc = extract_acc(baseline, TASK)
rsparse_acc  = extract_acc(rsparse,  TASK)
baseline_tps = baseline['latency']['tokens_per_sec']
rsparse_tps  = rsparse ['latency']['tokens_per_sec']

row = {
    'model'                   : MODEL_ALIAS,
    f'baseline acc ({TASK})'  : baseline_acc,
    f'r-sparse acc ({TASK})'  : rsparse_acc,
    'Δ acc'                   : (rsparse_acc - baseline_acc) if baseline_acc is not None and rsparse_acc is not None else None,
    'baseline tok/s'          : baseline_tps,
    'r-sparse tok/s'          : rsparse_tps,
    'speedup'                 : (rsparse_tps / baseline_tps) if baseline_tps else None,
}
df = pd.DataFrame([row])
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(df)

summary_path = RESULTS_DIR / 'summary.json'
summary_path.write_text(json.dumps({'task': TASK, 'rows': [row]}, indent=2, default=str))
print(f'\nWrote {summary_path}')

## §17 (Optional) vLLM latency reference

Curious how much faster vLLM would be on the same model? This cell loads the unmodified model into vLLM and times the same prompt batch. We don't use vLLM for accuracy here because the in-process vLLM `LLM` instance is hard to fully clean up inside a notebook — it holds GPU memory until process exit. If you skip §15-§16 and only run §17, you avoid the cleanup issue.

In [ ]:
# Make sure no other model is still holding the GPU.
import gc; gc.collect(); torch.cuda.empty_cache()
print(f'Free VRAM before vLLM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB')

from vllm import LLM, SamplingParams

llm = LLM(
    model=HF_ID,
    dtype='float16',
    gpu_memory_utilization=0.85,
    trust_remote_code=True,
    download_dir=str(HF_CACHE),
)
sampling = SamplingParams(temperature=0.0, max_tokens=LATENCY_MAX_TOKENS)
llm.generate(LATENCY_PROMPTS[:1], sampling)   # warm-up

t0 = time.perf_counter()
outputs = llm.generate(LATENCY_PROMPTS, sampling)
elapsed = time.perf_counter() - t0
total = sum(len(o.outputs[0].token_ids) for o in outputs)
vllm_tps = total / elapsed

print(f'\nvLLM latency: {vllm_tps:.2f} tok/s  ({total} tokens in {elapsed:.2f} s)')
vllm_path = RESULTS_DIR / f'{MODEL_ALIAS}_vllm_latency.json'
vllm_path.write_text(json.dumps({
    'alias': MODEL_ALIAS, 'hf_id': HF_ID, 'method': 'vllm_latency_only',
    'tokens_per_sec': vllm_tps, 'total_output_tokens': total, 'wall_clock_s': elapsed,
}, indent=2))
print(f'Wrote {vllm_path}')

## §18 (Optional) Sweep `target_sparsity`

Re-run R-Sparse at multiple sparsity levels to plot the accuracy / speedup curve. Each iteration reuses the SVD `.pt` files and the baseline JSON — only the R-Sparse evaluation runs again.

In [ ]:
sweep_rows = []
for ts in [0.3, 0.5, 0.7]:
    print(f'\n========== target_sparsity = {ts} ==========')
    res = run_method('r_sparse', target_sparsity=ts, sparse_ratio=SPARSE_RATIO, prefill_ratio=PREFILL_RATIO)
    sweep_rows.append({
        'target_sparsity' : ts,
        'acc'             : extract_acc(res, TASK),
        'tok/s'           : res['latency']['tokens_per_sec'],
    })

sweep_df = pd.DataFrame(sweep_rows)
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(sweep_df)
(RESULTS_DIR / f'{MODEL_ALIAS}_sweep.json').write_text(
    json.dumps({'task': TASK, 'rows': sweep_rows}, indent=2, default=str))